In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [2]:
from datetime import datetime
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
# 🔐 Load API keys

import os
from dotenv import load_dotenv

# ✅ Step 2: Load API key
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")

prompt = PromptTemplate.from_template("Translate to Tamil: {text}")
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", temperature=0)
#parser = StrOutputParser()

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"text": "Good morning"})
print(response)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Good morning in Tamil is:

**காலை வணக்கம்** 
(Pronunciation: Kaalai Vanakkam)


In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

prompt = PromptTemplate.from_template("List 5 programming languages, comma-separated.")
parser = CommaSeparatedListOutputParser()
chain = prompt | llm | parser

response = chain.invoke({})
print(response)  # ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']


['Python', 'JavaScript', 'Java', 'C++', 'Ruby']


In [4]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

parser = PydanticOutputParser(pydantic_object=ProductInfo)

prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "The Redmi Note 12 is available for ₹14,999."
})
print(response)


name='Redmi Note 12' price=14999.0


In [5]:
#from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

schemas = [
    ResponseSchema(name="company", description="Name of the company"),
    ResponseSchema(name="founder", description="Name of the founder"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

prompt = PromptTemplate(
    template="Extract company and founder from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})
print(response)


{'company': 'Hope AI', 'founder': 'Ramisha Rani'}
